# Lab Assignment 2 - Part B: k-Nearest Neighbor Classification
Please refer to the `README.pdf` for full laboratory instructions.


## Problem Statement
In this part, you will implement the k-Nearest Neighbor (k-NN) classifier and evaluate it on two datasets:
- **Lenses Dataset**: A small dataset for contact lens prescription
- **Credit Approval (CA) Dataset**: Credit card application data with binary labels (+/-)

### Your Tasks
1. **Preprocess the data**: Handle missing values and normalize features
2. **Implement k-NN** with L2 distance
3. **Evaluate** on both datasets for different values of k
4. **Discuss** your results

### Datasets
The data files are located in the `credit 2017/` folder:
- `lenses.training`, `lenses.testing`
- `crx.data.training`, `crx.data.testing`
- `crx.names` (describes the features)


## Setup


In [73]:
# Library declarations
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


In [74]:
# Data paths
DATA_PATH = "credit 2017/"

# Load Lenses data
def load_lenses_data():
    """Load the lenses dataset."""
    train_data = np.loadtxt(DATA_PATH + "lenses.training", delimiter=',')
    test_data = np.loadtxt(DATA_PATH + "lenses.testing", delimiter=',')
    
    # First column is ID, last column is label
    X_train = train_data[:, 1:-1]
    y_train = train_data[:, -1]
    X_test = test_data[:, 1:-1]
    y_test = test_data[:, -1]
    
    return X_train, y_train, X_test, y_test

# Load Credit Approval data
def load_credit_data():
    """
    Load the Credit Approval dataset.
    Note: This dataset contains missing values (?) and mixed types.
    You will need to preprocess it.
    """
    # TODO: Implement data loading
    # The data is comma-separated
    # Missing values are marked with '?'
    # Last column is the label ('+' or '-')
    train = pd.read_csv(DATA_PATH + "crx.data.training", header=None, na_values='?')
    test  = pd.read_csv(DATA_PATH + "crx.data.testing",  header=None, na_values='?')

    # labels
    y_train = train.pop(15).map({'+': 1, '-': 0}).values
    y_test  = test.pop(15).map({'+': 1, '-': 0}).values

    # impute missing
    for c in train.columns:
        if train[c].dtype == object:
            train[c] = train[c].fillna(train[c].mode()[0])
            test[c]  = test[c].fillna(train[c].mode()[0])
        else:
            train[c] = train[c].fillna(train[c].mean())
            test[c]  = test[c].fillna(train[c].mean())

    train = pd.get_dummies(train)
    test  = pd.get_dummies(test)
    test  = test.reindex(columns=train.columns, fill_value=0)

    num_cols = [c for c in train.columns if train[c].dtype != object]
    for c in num_cols:
        mu, sigma = train[c].mean(), train[c].std()
        train[c] = (train[c] - mu) / sigma
        test[c]  = (test[c]  - mu) / sigma

    return train.values, y_train, test.values, y_test
    

# Test loading lenses data
X_train_lenses, y_train_lenses, X_test_lenses, y_test_lenses = load_lenses_data()
print(f"Lenses - Train: {X_train_lenses.shape}, Test: {X_test_lenses.shape}")


Lenses - Train: (18, 3), Test: (6, 3)


## Task 1: Data Preprocessing
For the Credit Approval dataset, you need to:
1. **Handle missing values** (marked with '?'):
   - Categorical features: replace with mode/median
   - Numerical features: replace with label-conditioned mean
2. **Normalize features** using z-scaling:
   $$z_i^{(m)} = \frac{x_i^{(m)} - \mu_i}{\sigma_i}$$

Document exactly how you handle each feature!


In [75]:
def preprocess_credit_data(train_file, test_file):
    """
    Preprocess the Credit Approval dataset.
    
    Steps:
    1. Load the data
    2. Handle missing values
    3. Encode categorical variables
    4. Normalize numerical features
    
    Returns:
    --------
    X_train, y_train, X_test, y_test : numpy arrays
    """
    # TODO: Implement preprocessing
    # Hint: Read crx.names to understand the features
    # Feature types (from crx.names):
    # A1: categorical (b, a)
    # A2: continuous
    # A3: continuous
    # A4: categorical (u, y, l, t)
    # A5: categorical (g, p, gg)
    # A6: categorical (c, d, cc, i, j, k, m, r, q, w, x, e, aa, ff)
    # A7: categorical (v, h, bb, j, n, z, dd, ff, o)
    # A8: continuous
    # A9: categorical (t, f)
    # A10: categorical (t, f)
    # A11: continuous
    # A12: categorical (t, f)
    # A13: categorical (g, p, s)
    # A14: continuous
    # A15: continuous
    train = np.genfromtxt(train_file, delimiter=',', dtype=str)
    test  = np.genfromtxt(test_file,  delimiter=',', dtype=str)

    y_train = (train[:, -1] == '+').astype(int)
    y_test  = (test[:, -1]  == '+').astype(int)
    train, test = train[:, :-1], test[:, :-1]

    num_cols = [1, 2, 7, 10, 13, 14]
    cat_cols = [c for c in range(train.shape[1]) if c not in num_cols]

    for c in cat_cols:
        mode = max(set(train[:, c][train[:, c] != '?'].tolist()),
                key=train[:, c][train[:, c] != '?'].tolist().count)
        train[:, c] = np.where(train[:, c] == '?', mode, train[:, c])
        test[:, c]  = np.where(test[:, c]  == '?', mode, test[:, c])

    for c in num_cols:
        mean = np.array(train[:, c][train[:, c] != '?'], dtype=float).mean()
        train[:, c] = np.where(train[:, c] == '?', str(mean), train[:, c])
        test[:, c]  = np.where(test[:, c]  == '?', str(mean), test[:, c])

    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(np.unique(train[:, c]))}
        train[:, c] = [mapping.get(v, 0) for v in train[:, c]]
        test[:, c]  = [mapping.get(v, 0) for v in test[:, c]]

    X_train, X_test = z_normalize(train.astype(float), test.astype(float), num_cols)
    return X_train, y_train, X_test, y_test


def z_normalize(X_train, X_test, feature_indices):
    """
    Apply z-score normalization to specified features.
    
    Parameters:
    -----------
    X_train, X_test : numpy arrays
    feature_indices : list of indices for numerical features
    
    Returns:
    --------
    X_train_normalized, X_test_normalized : numpy arrays
    """
    # TODO: Implement z-normalization
    X_train, X_test = X_train.astype(float), X_test.astype(float)
    for i in feature_indices:
        mu, sigma = X_train[:, i].mean(), X_train[:, i].std()
        X_train[:, i] = (X_train[:, i] - mu) / (sigma + 1e-8)
        X_test[:, i]  = (X_test[:, i]  - mu) / (sigma + 1e-8)
    return X_train, X_test


## Task 2: Implement k-NN Classifier
Implement k-NN with L2 (Euclidean) distance:
$$\mathcal{D}_{L2}(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_i (a_i - b_i)^2}$$

For **categorical attributes**, use:
- Distance = 1 if values are different
- Distance = 0 if values are the same


In [76]:
def l2_distance(a, b):
    """
    Compute L2 (Euclidean) distance between two vectors.
    
    Parameters:
    -----------
    a, b : numpy arrays of same shape
    
    Returns:
    --------
    distance : float
    """
    # TODO: Implement L2 distance
    return np.sqrt(np.sum((a - b) ** 2))


def knn_predict(X_train, y_train, X_test, k):
    """
    Predict labels for test data using k-NN.
    
    Parameters:
    -----------
    X_train : numpy array of shape (n_train, n_features)
    y_train : numpy array of shape (n_train,)
    X_test : numpy array of shape (n_test, n_features)
    k : int, number of neighbors
    
    Returns:
    --------
    predictions : numpy array of shape (n_test,)
    """
    # TODO: Implement k-NN prediction
    # For each test sample:
    #   1. Compute distance to all training samples
    #   2. Find k nearest neighbors
    #   3. Predict using majority voting
    predictions = []
    for x in X_test:
        dists   = np.array([l2_distance(x, xt) for xt in X_train])
        k_idx   = np.argsort(dists)[:k]
        k_labels = y_train[k_idx]
        predictions.append(np.bincount(k_labels.astype(int)).argmax())
    return np.array(predictions)



def compute_accuracy(y_true, y_pred):
    """
    Compute classification accuracy.
    
    Returns:
    --------
    accuracy : float (between 0 and 1)
    """
    # TODO: Implement accuracy computation
    return np.mean(y_true == y_pred)


## Task 3: Evaluate on Lenses Dataset
Test your k-NN implementation on the Lenses dataset for different values of k.


In [77]:
# TODO: Evaluate k-NN on Lenses dataset
# Try different values of k (e.g., 1, 3, 5, 7)

k_values = [1, 3, 5, 7]
lenses_results = []

for k in k_values:
    predictions = knn_predict(X_train_lenses, y_train_lenses, X_test_lenses, k)
    accuracy = compute_accuracy(y_test_lenses, predictions)
    lenses_results.append((k, accuracy))
    print(f"k={k}: Accuracy = {accuracy:.4f}")


k=1: Accuracy = 1.0000
k=3: Accuracy = 1.0000
k=5: Accuracy = 0.5000
k=7: Accuracy = 0.8333


## Task 4: Evaluate on Credit Approval Dataset
First preprocess the data, then evaluate k-NN.


In [78]:
# TODO: Preprocess Credit Approval data

X_train_credit, y_train_credit, X_test_credit, y_test_credit = preprocess_credit_data(
    DATA_PATH + "crx.data.training",
    DATA_PATH + "crx.data.testing"
)
print(f"Credit - Train: {X_train_credit.shape}, Test: {X_test_credit.shape}")


Credit - Train: (552, 15), Test: (138, 15)


In [79]:
# TODO: Evaluate k-NN on Credit Approval dataset
k_values = [1, 3, 5, 7]
credit_results = []

for k in k_values:
    predictions = knn_predict(X_train_credit, y_train_credit, X_test_credit, k)
    accuracy = compute_accuracy(y_test_credit, predictions)
    credit_results.append((k, accuracy))
    print(f"k={k}: Accuracy = {accuracy:.4f}")


k=1: Accuracy = 0.7464
k=3: Accuracy = 0.7826
k=5: Accuracy = 0.8043
k=7: Accuracy = 0.7971


## Summary and Discussion

### Results Table

| Dataset | k=1 | k=3 | k=5 | k=7 |
|---------|-----|-----|-----|-----|
| Lenses | 1.0000 | 1.0000 | 0.5000 | 0.8333 |
| Credit Approval | 0.7464| 0.7826 | 0.8043 |  0.7971|

### Discussion
*Answer these questions:*
1. Which value of k works best for each dataset? Why do you think that is?
For Lenses 1 and 3, both got perfect accuracy but the test set is only 6 samples so it doesn't mean much. For Credit 5 was best at 80.43%, so lower k overfits noise, higher k over-smooths.
2. How did preprocessing affect your results on the Credit Approval dataset?
Z-normalization was important for Credit since features like A15 have large magnitudes and would dominate L2 distance without scaling 
Mean/mode imputation also helped avoid losing rows with missing values
3. What are the trade-offs of using different values of k?
- Low k = high variance, sensitive to noisy neighbors
- High k = high bias, majority class takes over
- k=5 was the best for Credit
4. What did you learn from this exercise?
- Preprocessing mattered more than the choice of k
- Scaling and imputation had a bigger effect on accuracy than tuning k
